In [13]:
import os
import pickle
import zipfile
import random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from shutil import copyfile
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    BatchNormalization, Conv2D, MaxPooling2D, Activation, Flatten, Dropout, Dense
)
from tensorflow.keras import backend as K
from sklearn.preprocessing import LabelBinarizer
from keras.preprocessing import image
import matplotlib.pyplot as plt

In [14]:
class DistillerEnsemble(keras.Model):
    def __init__(self, students, teacher):
        super(DistillerEnsemble, self).__init__()
        self.teacher = teacher
        self.students = students

    def compile(
        self,
        optimizer,
        metrics,
        student_loss_fn,
        distillation_loss_fn,
        alpha=0.1,
        temperature=3,
    ):
        """ Configure the distiller.

        Args:
            optimizer: Keras optimizer for the student weights
            metrics: Keras metrics for evaluation
            student_loss_fn: Loss function of difference between student
                predictions and ground-truth
            distillation_loss_fn: Loss function of difference between soft
                student predictions and soft teacher predictions
            alpha: weight to student_loss_fn and 1-alpha to distillation_loss_fn
            temperature: Temperature for softening probability distributions.
                Larger temperature gives softer distributions.
        """
        super(DistillerEnsemble, self).compile(optimizer=optimizer, metrics=metrics)
        self.student_loss_fn = student_loss_fn
        self.distillation_loss_fn = distillation_loss_fn
        self.alpha = alpha
        self.temperature = temperature

    def train_step(self, data):
        # Unpack data
        x, y = data

        # Forward pass of teacher
        teacher_predictions = self.teacher(x, training=False)

        with tf.GradientTape() as tape:
            total_loss = 0
            for student in self.students:
                # Forward pass of student
                student_predictions = student(x, training=True)

                # Compute losses
                student_loss = self.student_loss_fn(y, student_predictions)
                distillation_loss = self.distillation_loss_fn(
                    tf.nn.softmax(teacher_predictions / self.temperature, axis=1),
                    tf.nn.softmax(student_predictions / self.temperature, axis=1),
                )
                loss = self.alpha * student_loss + (1 - self.alpha) * distillation_loss
                total_loss += loss

        # Compute gradients
        trainable_vars = sum([student.trainable_variables for student in self.students], [])
        gradients = tape.gradient(total_loss, trainable_vars)

        # Update weights
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))

        # Update the metrics configured in `compile()`.
        self.compiled_metrics.update_state(y, student_predictions)

        # Return a dict of performance
        results = {m.name: m.result() for m in self.metrics}
        results.update(
            {"student_loss": student_loss, "distillation_loss": distillation_loss}
        )
        return results

    def test_step(self, data):
        # Unpack the data
        x, y = data

        # Compute predictions
        y_predictions = [student(x, training=False) for student in self.students]

        # Calculate the loss for each student and ensemble them
        student_losses = [self.student_loss_fn(y, y_pred) for y_pred in y_predictions]
        ensemble_loss = tf.reduce_mean(student_losses)

        # Update the metrics.
        self.compiled_metrics.update_state(y, tf.reduce_mean(y_predictions, axis=0))
        
        # Return a dict of performance
        results = {m.name: m.result() for m in self.metrics}
        results.update({"ensemble_loss": ensemble_loss})
        return results

    def call(self, data, training=False): 
        # You don't need this method for training.
        # So just pass.
        pass




In [15]:
# Assuming you've defined the function plotImages(images_arr) and created train_datagen and validation_datagen
def plotImages(images_arr):
    fig, axes = plt.subplots(1, 5, figsize=(20,20))
    axes = axes.flatten()
    for img, ax in zip(images_arr, axes):
        ax.imshow(img)
    plt.tight_layout()
    plt.show()

# Create data generators
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

validation_datagen = ImageDataGenerator(rescale=1.0/255)


In [16]:
# Split data into train, validation, and test sets
import splitfolders
splitfolders.ratio('plantvillage-dataset/color', output="output", seed=1337, ratio=(.8, 0.1, 0.1))

# Load data generators
TRAINING_DIR = "./output/train"
train_datagen = ImageDataGenerator(rescale=1.0/255)
train_generator = train_datagen.flow_from_directory(TRAINING_DIR,
                                                    class_mode='categorical',
                                                    batch_size=500,
                                                    target_size=(250, 250))

VALIDATION_DIR = "./output/test"
validation_datagen = ImageDataGenerator(rescale=1.0/255.)
validation_generator = validation_datagen.flow_from_directory(VALIDATION_DIR,
                                                              class_mode='categorical',
                                                              batch_size=500,
                                                              target_size=(250, 250))


Found 43429 images belonging to 38 classes.
Found 5459 images belonging to 38 classes.


In [17]:
# Assuming you've defined the function create_teacher_model(input_shape, num_classes)
def create_teacher_model(input_shape, num_classes):
    model = Sequential(name="teacher")
    chanDim = -1
    if K.image_data_format() == "channels_first":
        inputShape = (3, 250, 250)
        chanDim = 1
    model.add(Conv2D(32, (3, 3), padding="same", input_shape=input_shape))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chanDim))
    model.add(MaxPooling2D(pool_size=(3, 3)))
    model.add(Dropout(0.25))
    model.add(Conv2D(64, (3, 3), padding="same"))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chanDim))
    model.add(Conv2D(64, (3, 3), padding="same"))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chanDim))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))
    model.add(Conv2D(128, (3, 3), padding="same"))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chanDim))
    model.add(Conv2D(128, (3, 3), padding="same"))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chanDim))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))
    model.add(Flatten())
    model.add(Dense(1024))
    model.add(Activation("relu"))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))
    model.add(Dense(num_classes))
    return model


In [18]:
# Define the function to create a student model
def create_student_model(input_shape, num_classes):
    model = Sequential([
        Conv2D(32, (3, 3), padding="same", input_shape=input_shape),
        Activation("relu"),
        BatchNormalization(),
        MaxPooling2D(pool_size=(3, 3)),
        Dropout(0.25),
        Conv2D(64, (3, 3), padding="same"),
        Activation("relu"),
        BatchNormalization(),
        Conv2D(64, (3, 3), padding="same"),
        Activation("relu"),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),
        Conv2D(128, (3, 3), padding="same"),
        Activation("relu"),
        BatchNormalization(),
        Conv2D(128, (3, 3), padding="same"),
        Activation("relu"),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),
        Flatten(),
        Dense(1024),
        Activation("relu"),
        BatchNormalization(),
        Dropout(0.5),
        Dense(num_classes)
    ], name="student")
    return model

In [19]:
# Create the ensemble of student models
n_classes = 38  # Replace 10 with the actual number of classes in your dataset

num_students = 3  # Number of student models in the ensemble
students = [create_student_model((250, 250, 3), n_classes) for _ in range(num_students)]


In [20]:
# Create the teacher model
teacher_model = create_teacher_model((250, 250, 3), n_classes)

In [21]:
# Compile and train the teacher model
teacher_model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['acc']
)


In [22]:
teacher_history = teacher_model.fit(train_generator, validation_data=validation_generator, epochs=5)



Epoch 1/5
87/87 [==============================] - 2750s 32s/step - loss: 0.9007 - acc: 0.7548 - val_loss: 9.7912 - val_acc: 0.0181
Epoch 2/5
87/87 [==============================] - 2593s 30s/step - loss: 0.3127 - acc: 0.9032 - val_loss: 8.9664 - val_acc: 0.0432
Epoch 3/5
87/87 [==============================] - 2525s 29s/step - loss: 0.1863 - acc: 0.9417 - val_loss: 9.8767 - val_acc: 0.1035
Epoch 4/5
87/87 [==============================] - 3259s 38s/step - loss: 0.1157 - acc: 0.9652 - val_loss: 10.6497 - val_acc: 0.2178
Epoch 5/5
87/87 [==============================] - 3232s 37s/step - loss: 0.0738 - acc: 0.9779 - val_loss: 9.2505 - val_acc: 0.2898


In [23]:
# Assuming you've defined the function plot_training_history(history)
def plot_training_history(history):
    # Plot training & validation accuracy values
    plt.plot(history.history['acc'])
    plt.plot(history.history['val_acc'])
    plt.title('Model accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')
    plt.show()

    # Plot training & validation loss values
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')
    plt.show()


In [24]:
# Distill teacher to students ensemble
distiller_ensemble = DistillerEnsemble(students=students, teacher=teacher_model)
distiller_ensemble.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(),
    metrics=['acc'],
    student_loss_fn=keras.losses.CategoricalCrossentropy(from_logits=True),
    distillation_loss_fn=keras.losses.KLDivergence(),
    alpha=0.1,
    temperature=10
)

In [ ]:
student_history = distiller_ensemble.fit(train_generator, validation_data=validation_generator, epochs=5)


In [ ]:
# Assuming you've defined the function create_student_model_from_scratch(input_shape, num_classes)
def create_student_model_from_scratch(input_shape, num_classes):
    model = Sequential([
        layers.Conv2D(16, (3, 3), strides=(2, 2), padding="same", input_shape=input_shape),
        layers.LeakyReLU(alpha=0.2),
        layers.MaxPooling2D(pool_size=(2, 2), strides=(1, 1), padding="same"),
        layers.Conv2D(32, (3, 3), strides=(2, 2), padding="same"),
        layers.LeakyReLU(alpha=0.2),
        layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding="same"),
        layers.Flatten(),
        layers.Dense(num_classes)
    ], name="student")
    return model


In [ ]:
# Compile and train each student model from scratch
student_models_scratch = [create_student_model_from_scratch((250, 250, 3), n_classes) for _ in range(num_students)]

for i, student_model_scratch in enumerate(student_models_scratch):
    student_model_scratch.compile(
        optimizer=keras.optimizers.Adam(),
        loss=keras.losses.CategoricalCrossentropy(from_logits=True),
        metrics=['acc']
    )
    student_history_scratch = student_model_scratch.fit(train_generator, epochs=5, validation_data=validation_generator)
    # Assuming you've defined the function plot_training_history(history)

In [ ]:
# Plotting training history
plot_training_history(student_history)


# Evaluate the ensemble of student models
evaluation_ensemble = distiller_ensemble.evaluate(validation_generator)
print("\nEvaluation of Ensemble of Student Models:")
print("Loss:", evaluation_ensemble[0])
print("Accuracy:", evaluation_ensemble[1])

# Plotting training history for student models trained from scratch
for i, student_history_scratch in enumerate(student_history_scratch):
    print("\nStudent Model {} Trained from Scratch:".format(i+1))
    plot_training_history(student_history_scratch)

    # Evaluate the student model trained from scratch
    evaluation_scratch = student_models_scratch[i].evaluate(validation_generator)
    print("\nEvaluation of Student Model {} Trained from Scratch:".format(i+1))
    print("Loss:", evaluation_scratch[0])
    print("Accuracy:", evaluation_scratch[1])
